In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("TradeCorpETL").getOrCreate()

In [6]:
df_orders = spark.read.parquet("../data/tmp/orders")
df_products = spark.read.parquet("../data/tmp/products")
df_customers = spark.read.parquet("../data/tmp/customers")
df_employees = spark.read.parquet("../data/tmp/employees")
df_order_details = spark.read.parquet("../data/tmp/order_details")
df_categories = spark.read.parquet("../data/tmp/categories")
df_shippers = spark.read.parquet("../data/tmp/shippers")
df_suppliers = spark.read.parquet("../data/tmp/suppliers")

In [ ]:
# Q21

In [7]:
df_orders_customers = df_orders.join(df_customers, on="customer_id", how="inner").select("order_id","company_name","country","order_date","freight")

In [8]:
df_orders_customers.show(2)

+--------+--------------------+-------+----------+-------+
|order_id|        company_name|country|order_date|freight|
+--------+--------------------+-------+----------+-------+
|   10400|  Eastern Connection|     UK|1997-01-01|  83.93|
|   10401|Rattlesnake Canyo...|    USA|1997-01-01|  12.51|
+--------+--------------------+-------+----------+-------+
only showing top 2 rows



In [ ]:
# Q22

In [9]:
df_products.show(2)

+----------+--------------------+-----------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+
|product_id|        product_name|supplier_id|category_id|  quantity_per_unit|unit_price|units_in_stock|units_on_order|reorder_level|discontinued|en_stock|
+----------+--------------------+-----------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+
|         3|       Aniseed Syrup|          1|          2|12 - 550 ml bottles|      10.0|            13|            70|           25|           0|    true|
|         4|Chef Anton's Caju...|          2|          2|     48 - 6 oz jars|      22.0|            53|             0|            0|           0|    true|
+----------+--------------------+-----------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+
only showing top 2 rows



In [10]:
df_order_details_products = df_order_details.join(df_products.select("product_id","product_name","category_id","unit_price"), on="product_id", how="inner")

In [11]:
df_order_details_products.show(2)

+----------+--------+-------------+--------+--------+----------+--------------------+-----------+----------+
|product_id|order_id|prix_unitaire|quantite|discount|sous_total|        product_name|category_id|unit_price|
+----------+--------+-------------+--------+--------+----------+--------------------+-----------+----------+
|        11|   10248|         14.0|      12|     0.0|     168.0|      Queso Cabrales|          4|      21.0|
|        72|   10248|         34.8|       5|     0.0|     174.0|Mozzarella di Gio...|          4|      34.8|
+----------+--------+-------------+--------+--------+----------+--------------------+-----------+----------+
only showing top 2 rows



In [ ]:
# Q23

In [12]:
df_categories.show(2)

+-----------+-------------+--------------------+-------+
|category_id|category_name|         description|picture|
+-----------+-------------+--------------------+-------+
|          1|    Beverages|Soft drinks, coff...|     \x|
|          2|   Condiments|Sweet and savory ...|     \x|
+-----------+-------------+--------------------+-------+
only showing top 2 rows



In [13]:
df_products_categories = df_products.join(df_categories.select("category_id","category_name","description"), on="category_id", how="inner")

In [14]:
df_products_categories.show(2)

+-----------+----------+--------------------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|category_id|product_id|        product_name|supplier_id|  quantity_per_unit|unit_price|units_in_stock|units_on_order|reorder_level|discontinued|en_stock|category_name|         description|
+-----------+----------+--------------------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|          2|         3|       Aniseed Syrup|          1|12 - 550 ml bottles|      10.0|            13|            70|           25|           0|    true|   Condiments|Sweet and savory ...|
|          2|         4|Chef Anton's Caju...|          2|     48 - 6 oz jars|      22.0|            53|             0|            0|           0|    true|   Condiments|Sweet and savory ...|
+-----------+----------+--------------------+-----

In [ ]:
# Q24A

In [48]:
df_orders_enriched = df_orders\
.join(df_customers, on="customer_id", how="inner")\
.join(df_order_details, on="order_id", how="inner")\
.join(df_employees, on="employee_id", how="inner")\
.join(df_products_categories, on="product_id", how="inner")\
.join(df_suppliers, on="supplier_id", how="inner")\
.join(df_shippers, on="shipper_id", how="inner")

In [29]:
df_orders_enriched_dtypes = spark.createDataFrame(df_orders_enriched.dtypes,["name","type"])

In [30]:
from pyspark.sql.functions import col

In [31]:
df_orders_enriched_dtypes.groupBy("name").count().filter(col("count")>1).show()

+-------------+-----+
|         name|count|
+-------------+-----+
| company_name|    3|
| contact_name|    2|
|contact_title|    2|
|       region|    2|
|  postal_code|    2|
|         city|    3|
|      country|    3|
|      address|    2|
|          fax|    2|
|        phone|    3|
+-------------+-----+



In [ ]:
# Q24B

In [51]:
df_orders_enriched = df_orders\
.join(df_customers.withColumnRenamed("company_name","customer_company_name")\
                  .withColumnRenamed("contact_name","customer_contact_name")\
                  .withColumnRenamed("contact_title","customer_contact_title")\
                  .withColumnRenamed("address","customer_address")\
                  .withColumnRenamed("postal_code","customer_postal_code")\
                  .withColumnRenamed("region","customer_region")\
                  .withColumnRenamed("city","customer_city")\
                  .withColumnRenamed("country","customer_country")\
                  .withColumnRenamed("fax","customer_fax")\
                  .withColumnRenamed("phone","customer_phone"), on="customer_id", how="inner")\
.join(df_order_details, on="order_id", how="inner")\
.join(df_employees.withColumnRenamed("city","employee_city")\
                  .withColumnRenamed("country","employee_country"), on="employee_id", how="inner")\
.join(df_products_categories, on="product_id", how="inner")\
.join(df_suppliers.withColumnRenamed("company_name","supplier_company_name")\
                  .withColumnRenamed("contact_name","supplier_contact_name")\
                  .withColumnRenamed("contact_title","supplier_contact_title")\
                  .withColumnRenamed("address","supplier_address")\
                  .withColumnRenamed("postal_code","supplier_postal_code")\
                  .withColumnRenamed("region","supplier_region")\
                  .withColumnRenamed("city","supplier_city")\
                  .withColumnRenamed("country","supplier_country")\
                  .withColumnRenamed("fax","supplier_fax")\
                  .withColumnRenamed("phone","supplier_phone"), on="supplier_id", how="inner")\
.join(df_shippers.withColumnRenamed("company_name","shipper_company_name")\
                 .withColumnRenamed("phone","shipper_phone"), on="shipper_id", how="inner")

In [52]:
df_orders_enriched_dtypes = spark.createDataFrame(df_orders_enriched.dtypes,["name","type"])

In [53]:
df_orders_enriched_dtypes.groupBy("name").count().filter(col("count")>1).show()

+----+-----+
|name|count|
+----+-----+
+----+-----+



In [ ]:
# Q25

In [56]:
from pyspark.sql.functions import sum, round

In [58]:
df_orders_enriched.groupBy("customer_company_name").agg(round(sum("sous_total"),2).alias('CA')).orderBy(col("CA").desc()).show(10)

+---------------------+--------+
|customer_company_name|      CA|
+---------------------+--------+
|           QUICK-Stop|51682.74|
|   Save-a-lot Markets|40238.09|
|         Ernst Handel|39975.91|
|       M�re Paillarde|22871.07|
| Rattlesnake Canyo...| 17636.1|
|        Simons bistro|16232.42|
| Hungry Owl All-Ni...|14403.03|
|       Folk och f� HB|13200.92|
|     HILARION-Abastos|11799.74|
|   Berglunds snabbk�p|11758.92|
+---------------------+--------+
only showing top 10 rows



In [ ]:
# Q26

In [61]:
from pyspark.sql.functions import count_distinct

In [64]:
df_orders_enriched.groupBy("category_name").agg(round(sum("sous_total"),2).alias('CA'),count_distinct("product_id").alias("nb produits")).orderBy(col("CA").desc()).show()

+--------------+--------+-----------+
| category_name|      CA|nb produits|
+--------------+--------+-----------+
|Dairy Products|108086.9|          9|
|     Beverages|90368.64|          9|
|   Confections|82657.78|         13|
|       Seafood|66959.23|         12|
|    Condiments| 54995.0|         11|
|Grains/Cereals|51463.63|          6|
|       Produce|40992.09|          4|
|  Meat/Poultry|11017.17|          2|
+--------------+--------+-----------+



In [ ]:
# Q27

In [71]:
from pyspark.sql.functions import date_trunc, date_format

In [73]:
df_orders_enriched.groupBy(date_format(date_trunc("month",col("order_date")),"yyyy-MM").alias("mois")).agg(round(sum("sous_total"),2).alias('CA')).orderBy(col("mois")).show()

+-------+--------+
|   mois|      CA|
+-------+--------+
|1997-01|51487.51|
|1997-02|31549.04|
|1997-03|33226.33|
|1997-04| 41510.6|
|1997-05|48895.27|
|1997-06|29875.47|
|1997-07|45162.88|
|1997-08|38039.93|
|1997-09|43335.43|
|1997-10| 48574.5|
|1997-11|39898.78|
|1997-12| 54984.7|
+-------+--------+



In [ ]:
# Q28

In [80]:
from pyspark.sql.functions import mean, datediff

In [87]:
df_orders_enriched.groupBy("full_name").agg(count_distinct("order_id").alias("nb commandes"),round(sum("sous_total"),2).alias('CA'),round(mean(datediff("shipped_date","order_date")),1).alias('délais livraison nb jours')).orderBy(col("CA").desc()).show()

+----------------+------------+---------+-------------------------+
|       full_name|nb commandes|       CA|délais livraison nb jours|
+----------------+------------+---------+-------------------------+
|Margaret Peacock|          75|104193.78|                      8.3|
| Janet Leverling|          71| 97081.27|                      8.9|
|   Nancy Davolio|          54| 81898.92|                      7.8|
|   Andrew Fuller|          40| 54907.03|                     10.2|
|     Robert King|          33| 49562.78|                      9.8|
|  Laura Callahan|          53| 47077.95|                      8.0|
|  Michael Suyama|          33| 34037.07|                      7.9|
|  Anne Dodsworth|          18| 20595.99|                     10.0|
| Steven Buchanan|          18| 17185.65|                      6.5|
+----------------+------------+---------+-------------------------+



In [ ]:
# Q29

In [88]:
from pyspark.sql.window import Window

In [90]:
window_spec = Window.partitionBy("category_name").orderBy(col("CA").desc())

In [91]:
from pyspark.sql.functions import dense_rank

In [95]:
df_orders_enriched.groupBy("category_name","product_name").agg(round(sum("sous_total"),2).alias('CA')).withColumn("rang",dense_rank().over(window_spec)).show(100)

+--------------+--------------------+--------+----+
| category_name|        product_name|      CA|rang|
+--------------+--------------------+--------+----+
|     Beverages|       C�te de Blaye|49198.09|   1|
|     Beverages|         Ipoh Coffee| 11069.9|   2|
|     Beverages|        Lakkalik��ri|  7379.1|   3|
|     Beverages|       Outback Lager|  5468.4|   4|
|     Beverages|      Steeleye Stout|  5274.9|   5|
|     Beverages|Rh�nbr�u Klosterbier| 4485.55|   6|
|     Beverages|    Chartreuse verte|  4475.7|   7|
|     Beverages|       Sasquatch Ale|  2107.0|   8|
|     Beverages|Laughing Lumberja...|   910.0|   9|
|    Condiments|Louisiana Fiery H...|  9373.2|   1|
|    Condiments|      Sirop d'�rable|  9091.5|   2|
|    Condiments|        Vegie-spread| 6899.26|   3|
|    Condiments|        Gula Malacca| 6737.95|   4|
|    Condiments|Chef Anton's Caju...| 5214.88|   5|
|    Condiments|Original Frankfur...| 4761.38|   6|
|    Condiments|Northwoods Cranbe...|  4260.0|   7|
|    Condime

In [ ]:
# Q30

In [99]:
window_spec = Window.orderBy(col("mois"))

In [102]:
df_orders_enriched.groupBy(date_format(date_trunc("month",col("order_date")),"yyyy-MM").alias("mois")).agg(round(sum("sous_total"),2).alias('CA')).withColumn("CA cumulé",round(sum('CA').over(window_spec),2)).show()

+-------+--------+---------+
|   mois|      CA|    cumul|
+-------+--------+---------+
|1997-01|51487.51| 51487.51|
|1997-02|31549.04| 83036.55|
|1997-03|33226.33|116262.88|
|1997-04| 41510.6|157773.48|
|1997-05|48895.27|206668.75|
|1997-06|29875.47|236544.22|
|1997-07|45162.88| 281707.1|
|1997-08|38039.93|319747.03|
|1997-09|43335.43|363082.46|
|1997-10| 48574.5|411656.96|
|1997-11|39898.78|451555.74|
|1997-12| 54984.7|506540.44|
+-------+--------+---------+



In [ ]:
# Q31

In [107]:
df_orders_enriched.groupBy("product_name").agg(sum("quantite").alias("nb total vendu")).orderBy(col("nb total vendu").desc()).show(5)

+--------------------+--------------+
|        product_name|nb total vendu|
+--------------------+--------------+
|Gnocchi di nonna ...|           971|
|Raclette Courdavault|           752|
|   Camembert Pierrot|           665|
|Rh�nbr�u Klosterbier|           630|
| Sir Rodney's Scones|           610|
+--------------------+--------------+
only showing top 5 rows



In [108]:
df_orders_enriched.groupBy("customer_country").agg(round(sum("sous_total"),2).alias('CA')).orderBy(col("CA").desc()).show(3)

+----------------+---------+
|customer_country|       CA|
+----------------+---------+
|         GERMANY|100641.29|
|             USA| 90731.71|
|         AUSTRIA| 46559.49|
+----------------+---------+
only showing top 3 rows



In [ ]:
# Q32

In [109]:
df_orders_enriched.write.mode("overwrite").parquet("../data/output/orders_enriched")

In [110]:
df_orders_enriched.count()

893

In [111]:
df_orders_enriched.dtypes

[('shipper_id', 'int'),
 ('supplier_id', 'int'),
 ('product_id', 'int'),
 ('employee_id', 'int'),
 ('order_id', 'int'),
 ('customer_id', 'string'),
 ('order_date', 'date'),
 ('required_date', 'date'),
 ('shipped_date', 'date'),
 ('freight', 'double'),
 ('ship_name', 'string'),
 ('ship_address', 'string'),
 ('ship_city', 'string'),
 ('ship_region', 'string'),
 ('ship_postal_code', 'string'),
 ('ship_country', 'string'),
 ('is_shipped', 'boolean'),
 ('customer_company_name', 'string'),
 ('customer_contact_name', 'string'),
 ('customer_contact_title', 'string'),
 ('customer_address', 'string'),
 ('customer_city', 'string'),
 ('customer_region', 'string'),
 ('customer_postal_code', 'string'),
 ('customer_country', 'string'),
 ('customer_phone', 'string'),
 ('customer_fax', 'string'),
 ('prix_unitaire', 'double'),
 ('quantite', 'int'),
 ('discount', 'double'),
 ('sous_total', 'double'),
 ('first_name', 'string'),
 ('last_name', 'string'),
 ('title', 'string'),
 ('hire_date', 'date'),
 ('emp